# 🌿 Botanical AI: Cloud Training Notebook

Run this notebook in Google Colab to train your InceptionV3 model without downloading the 9GB dataset to your local computer! Google Colab provides a **free GPU** and has lightning-fast internet.

**Before you start:** Go to `Runtime` -> `Change runtime type` and select **T4 GPU**.

In [ ]:
!pip install kagglehub split-folders tensorflow

In [ ]:
import os
import shutil
import kagglehub
import splitfolders

print("Downloading the 9GB dataset directly to Google's cloud servers...")
path = kagglehub.dataset_download("aryashah2k/indian-medicinal-leaves-dataset")

def find_classes_dir(base_path):
    for root, dirs, files in os.walk(base_path):
        if len(dirs) > 10:
            return root
    return base_path

classes_dir = find_classes_dir(path)
output_dir = "data"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

print("Splitting dataset...")
splitfolders.ratio(classes_dir, output=output_dir, seed=42, ratio=(0.8, 0.2))
os.rename(os.path.join(output_dir, 'val'), os.path.join(output_dir, 'validation'))
print("Dataset ready!")

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import json

IMAGE_SIZE = (299, 299)
BATCH_SIZE = 32
EPOCHS = 20
TRAIN_DIR = 'data/train'
VAL_DIR = 'data/validation'
MODEL_SAVE_PATH = 'medical_plant_model.keras'

train_ds = tf.keras.utils.image_dataset_from_directory(TRAIN_DIR, shuffle=True, batch_size=BATCH_SIZE, image_size=IMAGE_SIZE, label_mode='categorical')
val_ds = tf.keras.utils.image_dataset_from_directory(VAL_DIR, shuffle=True, batch_size=BATCH_SIZE, image_size=IMAGE_SIZE, label_mode='categorical')
class_names = train_ds.class_names

labels_dict = {i: name for i, name in enumerate(class_names)}
with open('labels.json', 'w') as f:
    json.dump(labels_dict, f, indent=4)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.1),
])

base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(299, 299, 3))
base_model.trainable = False

inputs = tf.keras.Input(shape=(299, 299, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.inception_v3.preprocess_input(x)
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
outputs = Dense(len(class_names), activation='softmax')(x)

model = Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
    ModelCheckpoint(MODEL_SAVE_PATH, save_best_only=True, monitor='val_accuracy')
]

print("Starting AI Training on GPU...")
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)


In [ ]:
from google.colab import files

print("Downloading trained model and labels to your computer...")
files.download('medical_plant_model.keras')
files.download('labels.json')